In [86]:
import requests
import os
import re
from dotenv import load_dotenv
from urlextract import URLExtract
load_dotenv()

CLIENT_ID = os.getenv("GITHUB_CLIENT_ID")
CLIENT_SECRET = os.getenv("GITHUB_CLIENT_SECRET")
REPO_URL = "https://github.com/TonyLiu2004/Gatherly"

In [87]:
# GET code url (the code is in the redirected url)
get_code_url = f"https://github.com/login/oauth/authorize?client_id={CLIENT_ID}"
print(get_code_url)
CODE = "8e60993a64a179dc5b64"

https://github.com/login/oauth/authorize?client_id=Iv23liiVYP1oOher1RF2


In [88]:
def getAccessToken(code):
    data = {
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "code": code
        }
    headers = {"Accept": "application/json"}

    response = requests.post("https://github.com/login/oauth/access_token", data=data, headers=headers)
    token_data = response.json()
    access_token = token_data.get("access_token")
    return access_token

In [107]:
ACCESS_TOKEN = getAccessToken(CODE)

In [90]:
def get_all_files(repo_url, access_token):
    # 1. Parse the URL to get owner and repo name
    # Example: https://github.com/psf/requests
    parts = repo_url.rstrip('/').split('/')
    owner = parts[-2]
    repo = parts[-1]
    
    # 2. Define the API endpoint
    # We use 'main' as the default branch; change if necessary
    url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/main?recursive=1"
    
    # 3. Set up headers (Token is highly recommended to avoid rate limits)
    headers = {"Accept": "application/vnd.github+json"}
    if access_token:
        headers["Authorization"] = f"Bearer {access_token}"
    
    # 4. Make the request
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        
        # Filter for 'blob' (files) and ignore 'tree' (folders)
        files = [item['path'] for item in data['tree'] if item['type'] == 'blob']
        return files
    else:
        return f"Error: {response.status_code} - {response.text}"

In [91]:
def get_github_file_content(repo_url, file_path, access_token):
    parts = repo_url.rstrip('/').split('/')
    owner, repo = parts[-2], parts[-1]
    
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{file_path}"
    
    headers = {
        # This tells GitHub to send the raw text, not the JSON/Base64 object
        "Accept": "application/vnd.github.v3.raw", 
        "Authorization": f"token {access_token}"
    }
    
    response = requests.get(url, headers=headers)
    response.raise_for_status() 
        
    return response.text

In [102]:
def get_links(text):
    extractor = URLExtract()
    links = extractor.find_urls(text)
    return [link for link in links if link.startswith(('http://', 'https://'))]

In [103]:
all_files = get_all_files("https://github.com/TonyLiu2004/Gatherly", ACCESS_TOKEN)
# all_files

In [106]:
TARGET_EXTENSIONS = ('.md', '.txt', '.html', '.js')
TARGET_FILES = ('package.json', 'CONTRIBUTING', 'LICENSE')
for file in all_files:
    if file.endswith(TARGET_EXTENSIONS) or any(t in file for t in TARGET_FILES):
        content = get_github_file_content(REPO_URL, file, ACCESS_TOKEN)
        if not content:
            print("no content")
            continue
        # print(content)
        links = get_links(content)
        if len(links) != 0:
            print(file)
            print(links)

README.md
['https://classroom.github.com/assets/deadline-readme-button-22041afd0340ce965d47ae6ef1cefeee28c7c493a6346c4f15d667ab976d596c.svg', 'https://classroom.github.com/a/_KG6YNPd', 'https://classroom.github.com/assets/launch-codespace-2972f46106e565e64193e422d61a12cf1da4916b45550586e14ef0a7c637dd04.svg', 'https://classroom.github.com/open-in-codespaces?assignment_repo_id=20208102', 'https://github.com/CSCI-40500-Fall-2025/project-project-1/actions/workflows/node.js.yml/badge.svg', 'https://github.com/CSCI-40500-Fall-2025/project-project-1/actions/workflows/node.js.yml', 'https://github.com/CSCI-40500-Fall-2025/project-project-1/actions/workflows/deploy.yml/badge.svg', 'https://github.com/CSCI-40500-Fall-2025/project-project-1/actions/workflows/deploy.yml', 'https://www.figma.com/design/uYJhX81BXaZXSea7oK4Wjr/SWE-Proj?node-id=0-1&t=lr60PfD8MiXTsyvA-1', 'http://localhost:5173', 'http://localhost:5173', 'http://localhost:3000', 'https://github.com/user-attachments/assets/0caaa3f1-e40f